# Notebook 3: Llama-3.2-1B-Instruct + LoRA (fine-tuned LLM baseline)
**Kaggle: Accelerator = GPU T4 (one GPU is used), Internet ON, attach Dataset `nli-data`,
add a Kaggle Secret named `HF_TOKEN` (Add-ons -> Secrets) holding your Hugging Face token.**

Formulation: causal LM with a fixed text prompt. The training loss is cross-entropy on the single answer token
(first token of " entailment" / " neutral" / " contradiction") at the last prompt position. At inference we read the logits of
those 3 tokens at the last position and softmax over them: no free generation, no invalid outputs.
**The prompt in `build_prompt` is the one the zero/few-shot baseline must reuse unchanged.**

Protocol: base fp16 (T4 has no bf16), LoRA weights fp32, fp16 autocast + GradScaler. A short pilot (2 learning rates,
5K train examples, scored on 2K validation examples) picks the lr; the throughput measured in the pilot decides how many
examples fit in `MAX_HOURS` (default: all 100K). Test sets are used once, after training. Single seed (42), stated as a limitation.

In [1]:
import glob, json, math, os, subprocess, sys, time
# Kaggle ships an old torchao (0.10) that peft>=0.20 refuses to work with. We don't use torchao, so remove it BEFORE peft/transformers are imported.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
from pathlib import Path
import numpy as np, pandas as pd, torch
try:
    import peft
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True); import peft
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup, set_seed

hits = glob.glob(os.environ.get("NLI_ROOT", "/kaggle/input") + "/**/nli_common.py", recursive=True)
assert hits, "nli_common.py not found: attach the 'nli-data' Dataset to this notebook"
ROOT = Path(hits[0]).parent
sys.path.insert(0, str(ROOT))
DATA = str(ROOT / "data")
from nli_common import LABELS, SEED, load_split, save_run

OUT = Path(os.environ.get("NLI_OUT", "/kaggle/working/results/llama1b_lora")); OUT.mkdir(parents=True, exist_ok=True)
CKPT = Path("/kaggle/working/ckpt"); CKPT.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "Turn on a GPU accelerator (T4) in the notebook settings"
dev = "cuda"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| peft", peft.__version__)

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "Add a Kaggle Secret named HF_TOKEN (your Hugging Face token)"

MODEL = "meta-llama/Llama-3.2-1B-Instruct"
MICRO_BS, ACCUM = 16, 2                    # effective batch 32
LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
WARMUP, CLIP, WD = 0.03, 1.0, 0.0
LR_PILOT, PILOT_N, PILOT_VAL_N = [1e-4, 2e-4], 5000, 2000
N_FULL, EPOCHS, MAX_HOURS = 100_000, 1, 7.0
MAX_LEN, SAVE_EVERY = 512, 500

train = load_split("train_100k", DATA)
EVALS = {n: load_split(n, DATA) for n in ["snli_val", "snli_test", "mnli_m", "mnli_mm"]}
val = EVALS["snli_val"]
pilot_val = val.sample(PILOT_VAL_N, random_state=SEED).reset_index(drop=True)
print({"train": len(train), **{k: len(v) for k, v in EVALS.items()}})

GPU: Tesla T4 | torch 2.11.0+cu128 | peft 0.20.0
{'train': 100000, 'snli_val': 9842, 'snli_test': 9824, 'mnli_m': 9815, 'mnli_mm': 9832}


## Prompt, tokenisation, batching

In [2]:
tok = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN)
if "<|finetune_right_pad_id|>" in tok.get_vocab():
    tok.pad_token = "<|finetune_right_pad_id|>"
elif tok.pad_token is None:
    tok.pad_token = tok.eos_token
PAD = tok.pad_token_id

LABEL_WORDS = [" entailment", " neutral", " contradiction"]       # label 0, 1, 2
LABEL_IDS = [tok(w, add_special_tokens=False).input_ids[0] for w in LABEL_WORDS]
print("label first-tokens:", LABEL_IDS, [tok.decode([i]) for i in LABEL_IDS])
assert len(set(LABEL_IDS)) == 3, "label words must have distinct first tokens"
LABEL_IDS_T = torch.tensor(LABEL_IDS, device=dev)

def build_prompt(premise, hypothesis):      # SHARED with the zero/few-shot notebook: do not change one without the other
    return (f"Premise: {premise}\nHypothesis: {hypothesis}\n"
            "Does the premise entail the hypothesis, contradict it, or is it neutral?\n"
            "Answer with one word: entailment, neutral, or contradiction.\nAnswer:")

def encode_df(df):
    prompts = [build_prompt(p, h) for p, h in zip(df["premise"], df["hypothesis"])]
    seqs = tok(prompts, add_special_tokens=True).input_ids
    for i, s in enumerate(seqs):              # rare very long MNLI premises: shorten the PREMISE (never cut the question/answer cue)
        words = df["premise"].iloc[i].split()
        while len(s) > MAX_LEN and len(words) > 5:
            words = words[: int(len(words) * 0.85)]
            s = tok(build_prompt(" ".join(words), df["hypothesis"].iloc[i]), add_special_tokens=True).input_ids
        seqs[i] = s
    return seqs

def pad_left(seqs):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), PAD, dtype=torch.long); mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, L - len(s):] = torch.tensor(s); mask[i, L - len(s):] = 1
    pos = (mask.cumsum(1) - 1).clamp(min=0)   # position ids ignore left padding
    return ids.to(dev), mask.to(dev), pos.to(dev)

_KEEP = {"name": "logits_to_keep"}
def last_logits(model, ids, mask, pos):
    """Logits at the last position only (B, V). Avoids materialising (B, L, 128K) logits."""
    while True:
        try:
            kw = {_KEEP["name"]: 1} if _KEEP["name"] else {}
            return model(input_ids=ids, attention_mask=mask, position_ids=pos, **kw).logits[:, -1, :]
        except TypeError:
            if _KEEP["name"] is None:
                raise
            _KEEP["name"] = "num_logits_to_keep" if _KEEP["name"] == "logits_to_keep" else None

@torch.no_grad()
def predict(model, df, bs=32):
    model.eval()
    seqs = encode_df(df)
    order = np.argsort([len(s) for s in seqs])
    out = np.zeros((len(df), 3), dtype=np.float32)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        ids, mask, pos = pad_left([seqs[j] for j in idx])
        with torch.autocast("cuda", dtype=torch.float16):
            lg = last_logits(model, ids, mask, pos)
        out[idx] = torch.softmax(lg[:, LABEL_IDS_T].float(), -1).cpu().numpy()
    return out

def acc_of(model, df, bs=32):
    return float((predict(model, df, bs).argmax(1) == df["label"].to_numpy()).mean())

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


label first-tokens: [87092, 21277, 50859] [' entail', ' neutral', ' contradiction']


## Model and training loop

In [3]:
def new_lora_model():
    try:
        base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, token=HF_TOKEN)
    except TypeError:
        base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, token=HF_TOKEN)
    base.to(dev)
    model = get_peft_model(base, LoraConfig(**LORA))
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()           # LoRA weights in fp32, base stays fp16
    return model

def train_lora(lr, df, seed, tag, ckpt_path=None):
    set_seed(seed)
    model = new_lora_model()
    trainable = [p for p in model.parameters() if p.requires_grad]
    n_train_params, n_total = sum(p.numel() for p in trainable), sum(p.numel() for p in model.parameters())
    seqs, labels = encode_df(df), df["label"].to_numpy()
    n = len(df); perm = np.random.default_rng(seed).permutation(n)
    n_micro = math.ceil(n / MICRO_BS) * EPOCHS
    total_steps = math.ceil(n_micro / ACCUM)
    opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=WD)
    sched = get_cosine_schedule_with_warmup(opt, int(WARMUP * total_steps), total_steps)
    scaler = torch.cuda.amp.GradScaler()
    start_step = 0
    if ckpt_path is not None and Path(ckpt_path).exists():            # resume after a crash / restart
        st = torch.load(ckpt_path, map_location=dev)
        if st["tag"] == tag:
            for (nme, p) in model.named_parameters():
                if nme in st["lora"]: p.data.copy_(st["lora"][nme])
            opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
            start_step = st["step"]; print(f"[resume] {tag} from step {start_step}/{total_steps}")
    torch.cuda.reset_peak_memory_stats()
    model.train(); t0 = time.time(); step = start_step; loss_acc, n_acc, nan_skips = 0.0, 0, 0
    micro_list = [(e, m) for e in range(EPOCHS) for m in range(math.ceil(n / MICRO_BS))]
    for mi in range(start_step * ACCUM, n_micro):
        e, m = micro_list[mi]
        idx = perm[m * MICRO_BS:(m + 1) * MICRO_BS]
        ids, mask, pos = pad_left([seqs[j] for j in idx])
        target = torch.tensor([LABEL_IDS[labels[j]] for j in idx], device=dev)
        with torch.autocast("cuda", dtype=torch.float16):
            lg = last_logits(model, ids, mask, pos)
        loss = torch.nn.functional.cross_entropy(lg.float(), target)
        if not torch.isfinite(loss):
            nan_skips += 1
            assert nan_skips < 50, "too many non-finite losses: fp16 problem, tell me"
        else:
            scaler.scale(loss / ACCUM).backward()
            loss_acc += loss.item(); n_acc += 1
        if (mi + 1) % ACCUM == 0 or mi == n_micro - 1:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainable, CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
            if step % 100 == 0 or step == total_steps:
                el = time.time() - t0; done = (step - start_step) * ACCUM * MICRO_BS
                print(f"  [{tag}] step {step}/{total_steps}  loss={loss_acc / max(1, n_acc):.4f}  "
                      f"{done / el:.1f} samples/s  elapsed {el / 60:.1f} min  ETA {(total_steps - step) * el / (step - start_step) / 60:.1f} min")
                loss_acc, n_acc = 0.0, 0
            if ckpt_path is not None and step % SAVE_EVERY == 0:
                try:
                    torch.save({"tag": tag, "step": step, "opt": opt.state_dict(), "sched": sched.state_dict(),
                                "scaler": scaler.state_dict(),
                                "lora": {k: p.detach().cpu() for k, p in model.named_parameters() if p.requires_grad}}, ckpt_path)
                except Exception as ex:
                    print("checkpoint save failed:", ex)
    train_time = time.time() - t0
    n_done = (n_micro * MICRO_BS) if start_step == 0 else (n_micro - start_step * ACCUM) * MICRO_BS
    return model, {"train_time_s": train_time, "samples_per_s": min(n_done, n * EPOCHS) / train_time,
                   "peak_gpu_mem_gb": torch.cuda.max_memory_allocated() / 1e9, "nan_skips": nan_skips,
                   "n_trainable_params": n_train_params, "n_total_params": n_total, "n_train_examples": n, "steps": total_steps}

## 1. Pilot: two learning rates on 5K examples (scored on 2K validation examples)

In [4]:
pilot_df = train.iloc[:PILOT_N].reset_index(drop=True)
pilot, sps = [], []
for lr in LR_PILOT:
    model, info = train_lora(lr, pilot_df, SEED, f"pilot_lr{lr:g}")
    acc = acc_of(model, pilot_val)
    pilot.append({"lr": lr, "val_acc_2k": acc, "train_time_s": round(info["train_time_s"]), "samples_per_s": round(info["samples_per_s"], 1)})
    sps.append(info["samples_per_s"]); print(pilot[-1])
    del model; torch.cuda.empty_cache()
pilot_df_out = pd.DataFrame(pilot); pilot_df_out.to_csv(OUT / "llama_lora_pilot.csv", index=False)
BEST_LR = float(pilot_df_out.sort_values("val_acc_2k", ascending=False).iloc[0]["lr"])
# fit the final run into the time budget (all 100K unless throughput is very low)
fit_n = int(MAX_HOURS * 3600 * 0.85 * float(np.mean(sps)) / EPOCHS)
N_USE = min(N_FULL, fit_n)
print(f"\nSelected lr: {BEST_LR:g} | throughput ~{np.mean(sps):.1f} samples/s | projected full run: "
      f"{N_USE * EPOCHS / np.mean(sps) / 3600:.2f} h on {N_USE:,} examples" + ("" if N_USE == N_FULL else "  (REDUCED to fit MAX_HOURS: state this in the report)"))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

/tmp/ipykernel_23/3320305593.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_23/3320305593.py:52: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1


  [pilot_lr0.0001] step 100/157  loss=0.7936  33.8 samples/s  elapsed 1.6 min  ETA 0.9 min
  [pilot_lr0.0001] step 157/157  loss=0.3937  33.5 samples/s  elapsed 2.5 min  ETA 0.0 min
{'lr': 0.0001, 'val_acc_2k': 0.871, 'train_time_s': 150, 'samples_per_s': 33.4}


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

  [pilot_lr0.0002] step 100/157  loss=0.7109  32.3 samples/s  elapsed 1.7 min  ETA 0.9 min
  [pilot_lr0.0002] step 157/157  loss=0.3574  32.5 samples/s  elapsed 2.6 min  ETA 0.0 min
{'lr': 0.0002, 'val_acc_2k': 0.889, 'train_time_s': 155, 'samples_per_s': 32.4}

Selected lr: 0.0002 | throughput ~32.9 samples/s | projected full run: 0.85 h on 100,000 examples


## 2. Final training run (seed 42), then evaluation on all four sets

In [5]:
final_df = train.iloc[:N_USE].reset_index(drop=True)      # first N rows of the shuffled stratified 100K = nested, ~balanced
print("final train class counts:", final_df["label"].value_counts().sort_index().to_dict())
model, info = train_lora(BEST_LR, final_df, SEED, "final", ckpt_path=CKPT / "last.pt")
model.save_pretrained(OUT / "llama1b_lora_adapter")

config = {"model": MODEL, "train_file": "snli_train_100k (first N rows)", "n_train": N_USE, "epochs": EPOCHS, "seed": SEED, "lr": BEST_LR,
          "lr_pilot_grid": LR_PILOT, "pilot_n": PILOT_N, "lora": {k: str(v) for k, v in LORA.items()},
          "micro_batch": MICRO_BS, "grad_accum": ACCUM, "effective_batch": MICRO_BS * ACCUM, "scheduler": "cosine",
          "warmup_frac": WARMUP, "weight_decay": WD, "grad_clip": CLIP, "precision": "base fp16, LoRA fp32, fp16 autocast + GradScaler",
          "max_len": MAX_LEN, "label_words": LABEL_WORDS, "label_token_ids": LABEL_IDS, "prompt_example": build_prompt("<premise>", "<hypothesis>")}
for es, df in EVALS.items():
    torch.cuda.synchronize(); t0 = time.time()
    probs = predict(model, df, bs=64 if es.startswith("snli") else 32)
    torch.cuda.synchronize(); dt = time.time() - t0
    save_run(OUT, "llama1b_lora", es, df, probs, config, {**info, "inference_s": dt, "inference_examples_per_s": len(df) / dt})
if (CKPT / "last.pt").exists(): os.remove(CKPT / "last.pt")

final train class counts: {0: 33387, 1: 33268, 2: 33345}


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

/tmp/ipykernel_23/3320305593.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


  [final] step 100/3125  loss=1.0358  31.8 samples/s  elapsed 1.7 min  ETA 50.7 min
  [final] step 200/3125  loss=0.5030  32.1 samples/s  elapsed 3.3 min  ETA 48.6 min
  [final] step 300/3125  loss=0.4186  32.2 samples/s  elapsed 5.0 min  ETA 46.7 min
  [final] step 400/3125  loss=0.4358  32.3 samples/s  elapsed 6.6 min  ETA 45.0 min
  [final] step 500/3125  loss=0.4096  32.3 samples/s  elapsed 8.3 min  ETA 43.3 min
  [final] step 600/3125  loss=0.3878  32.3 samples/s  elapsed 9.9 min  ETA 41.7 min
  [final] step 700/3125  loss=0.3700  32.3 samples/s  elapsed 11.5 min  ETA 40.0 min
  [final] step 800/3125  loss=0.3886  32.3 samples/s  elapsed 13.2 min  ETA 38.4 min
  [final] step 900/3125  loss=0.3712  32.3 samples/s  elapsed 14.8 min  ETA 36.7 min
  [final] step 1000/3125  loss=0.3818  32.3 samples/s  elapsed 16.5 min  ETA 35.1 min
  [final] step 1100/3125  loss=0.3530  32.3 samples/s  elapsed 18.2 min  ETA 33.4 min
  [final] step 1200/3125  loss=0.3461  32.3 samples/s  elapsed 19.8 m

/usr/local/lib/python3.13/dist-packages/peft/utils/other.py:1496: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-6ade3d8f-62c6973606367c2b1f01e1a9;d251ab34-8b4e-4c55-8195-744b38cfbecd)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-1B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Llama-3.2-1B-Instruct.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:438: UserWarning: Could not find a config file in meta-llama/Llama-3.2-1B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


[llama1b_lora | snli_val] acc=0.9198 macroF1=0.9197 per-class F1=[0.929, 0.891, 0.939]
[llama1b_lora | snli_test] acc=0.9124 macroF1=0.9121 per-class F1=[0.921, 0.878, 0.937]
[llama1b_lora | mnli_m] acc=0.8269 macroF1=0.8246 per-class F1=[0.863, 0.762, 0.849]
[llama1b_lora | mnli_mm] acc=0.8282 macroF1=0.8252 per-class F1=[0.863, 0.757, 0.855]


## Summary

In [6]:
rows = [{"eval_set": es, **{k: round(json.load(open(OUT / f"results_llama1b_lora_{es}.json"))["metrics"][k], 4) for k in ("accuracy", "f1_macro")}} for es in EVALS]
print(pd.DataFrame(rows).to_string(index=False))
print("\ntrain time: %.1f min | %.1f samples/s | peak GPU mem %.1f GB | trainable params %.1fM of %.0fM" % (
    info["train_time_s"] / 60, info["samples_per_s"], info["peak_gpu_mem_gb"], info["n_trainable_params"] / 1e6, info["n_total_params"] / 1e6))
print("Files:", sorted(os.listdir(OUT)))

 eval_set  accuracy  f1_macro
 snli_val    0.9198    0.9197
snli_test    0.9124    0.9121
   mnli_m    0.8269    0.8246
  mnli_mm    0.8282    0.8252

train time: 51.7 min | 32.2 samples/s | peak GPU mem 8.0 GB | trainable params 11.3M of 1247M
Files: ['llama1b_lora_adapter', 'llama_lora_pilot.csv', 'preds_llama1b_lora_mnli_m.csv', 'preds_llama1b_lora_mnli_mm.csv', 'preds_llama1b_lora_snli_test.csv', 'preds_llama1b_lora_snli_val.csv', 'results_llama1b_lora_mnli_m.json', 'results_llama1b_lora_mnli_mm.json', 'results_llama1b_lora_snli_test.json', 'results_llama1b_lora_snli_val.json']
